Visualize postprocessing steps:


In [ ]:

#!/usr/bin/env python3
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
import os
import json
import numpy as np
import nibabel as nib
import cc3d
from pathlib import Path
from scipy.ndimage import distance_transform_edt

# ============================================================
# Paths
# ============================================================

img_root   = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTr"
pred_root  = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTr_wp"
colon_root = "/data/colon_cancer/totalseg/total"
prob_root  = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTr_wp"

# ============================================================
# Loading
# ============================================================

def load_nifti(path):
    img = nib.load(str(path))
    return img.get_fdata(), img.affine, img.header

def load_ct(path):
    return np.transpose(load_nifti(path)[0], (2, 1, 0))

def load_label(path):
    return np.transpose(load_nifti(path)[0], (2, 1, 0)).astype(np.uint8)

def load_fg_probs(npz_path):
    return np.load(npz_path)["probabilities"][1]

def print_annotation_slices(mask, step_name):
    """
    Prints z-slices that contain any foreground.
    """
    z_any = mask.any(axis=(1, 2))
    z_idx = np.where(z_any)[0]

    if len(z_idx) == 0:
        print(f"[{step_name}] No annotated slices")
    else:
        print(
            f"[{step_name}] Annotated slices: "
            f"{z_idx.tolist()} "
            f"(min={z_idx.min()}, max={z_idx.max()}, count={len(z_idx)})"
        )

# ============================================================
# Pipeline logic
# ============================================================

def components_touching_colon(pred_mask, colon_mask, connectivity=26):

    labeled, num = cc3d.connected_components(pred_mask, return_N=True, connectivity=connectivity)

    colon_mask_bool = colon_mask > 0
    keep_ids = []

    for comp_id in range(1, num + 1):
        component_voxels = (labeled == comp_id)
        if np.any(component_voxels & colon_mask_bool):
            keep_ids.append(comp_id)

    cleaned = np.isin(labeled, keep_ids).astype(np.uint8)
    return cleaned

def interpolate_missing_slices(mask):
    mask = mask.astype(bool)
    z_any = mask.any(axis=(1, 2))
    z_idx = np.where(z_any)[0]

    if len(z_idx) < 2:
        return mask.astype(np.uint8)

    dt = np.zeros_like(mask, dtype=np.float32)
    for z in range(mask.shape[0]):
        fg = mask[z]
        dt[z] = distance_transform_edt(~fg) - distance_transform_edt(fg)

    repaired = mask.copy()
    for z in range(z_idx[0] + 1, z_idx[-1]):
        if not z_any[z]:
            zp = z_idx[z_idx < z][-1]
            zn = z_idx[z_idx > z][0]
            w = (z - zp) / float(zn - zp)
            repaired[z] = ((1 - w) * dt[zp] + w * dt[zn]) < 0

    return repaired.astype(np.uint8)

def connected_components(mask):
    labeled = cc3d.connected_components(mask)
    return [(labeled == cid) for cid in range(1, labeled.max() + 1)]

def merge_components_by_z_overlap(components):
    if len(components) == 0:
        return []

    merged, used = [], [False]*len(components)
    z_ranges = [(np.where(c)[0].min(), np.where(c)[0].max()) for c in components]

    for i, (c, (z0, z1)) in enumerate(zip(components, z_ranges)):
        if used[i]:
            continue
        m = c.copy()
        used[i] = True
        for j, (c2, (zz0, zz1)) in enumerate(zip(components, z_ranges)):
            if used[j]:
                continue
            if not (zz1 < z0 or zz0 > z1):
                m |= c2
                used[j] = True
        merged.append(m)

    return merged

def keep_largest_component(mask):
    labeled = cc3d.connected_components(mask)
    if labeled.max() == 0:
        return mask
    sizes = cc3d.statistics(labeled)["voxel_counts"]
    return (labeled == (np.argmax(sizes[1:]) + 1)).astype(np.uint8)

def keep_highest_score_component(mask, fg_probs, merge_by_z=False):
    comps = connected_components(mask)

    if len(comps) == 0:
        print("[WARN] No components available for scoring")
        return mask

    if merge_by_z and len(comps) > 1:
        comps = merge_components_by_z_overlap(comps)

    if len(comps) == 0:
        print("[WARN] No components after z-merge")
        return mask

    scores = [
        fg_probs[c].mean() if c.any() else -np.inf
        for c in comps
    ]

    return comps[np.argmax(scores)].astype(np.uint8)

# ============================================================
# Visualization helpers
# ============================================================

def window_ct(ct, level=50, width=350):
    lo, hi = level - width/2, level + width/2
    return np.clip((ct - lo) / (hi - lo + 1e-6), 0, 1)

def visualize_step(ct, prev_mask, curr_mask, title, alpha=0.35):
    kept = curr_mask > 0
    removed = (prev_mask > 0) & (~kept)

    overlay = np.zeros_like(curr_mask, dtype=np.uint8)
    overlay[kept] = 1
    overlay[removed] = 2

    cmap = ListedColormap([
        [0, 0, 0, 0],
        [0, 1, 0, alpha],  # kept
        [1, 0, 0, alpha],  # removed
    ])

    def plot(z):
        plt.figure(figsize=(6, 6))
        plt.imshow(window_ct(ct[z]), cmap="gray", origin="lower")
        plt.imshow(overlay[z], cmap=cmap, origin="lower", vmin=0, vmax=2)
        plt.title(f"{title} | slice {z}")
        plt.axis("off")
        plt.show()

    slider = widgets.IntSlider(
        value=ct.shape[0] // 2,
        min=0,
        max=ct.shape[0] - 1,
        step=1,
        description="Slice",
        continuous_update=False,
    )

    display(slider, widgets.interactive_output(plot, {"z": slider}))

def visualize_colon_debug(ct, pred, colon, title="Colon mask debug", alpha=0.35):
    overlay = np.zeros_like(pred, dtype=np.uint8)
    overlay[colon > 0] = 1
    overlay[pred > 0] = 2
    overlay[(colon > 0) & (pred > 0)] = 3

    cmap = ListedColormap([
        [0, 0, 0, 0],
        [0, 0, 1, alpha],  # colon
        [0, 1, 0, alpha],  # prediction
        [1, 1, 0, alpha],  # overlap
    ])

    def plot(z):
        plt.figure(figsize=(6, 6))
        plt.imshow(window_ct(ct[z]), cmap="gray", origin="lower")
        plt.imshow(overlay[z], cmap=cmap, origin="lower", vmin=0, vmax=3)
        plt.title(f"{title} | slice {z}")
        plt.axis("off")
        plt.show()

    slider = widgets.IntSlider(
        value=ct.shape[0] // 2,
        min=0,
        max=ct.shape[0] - 1,
        step=1,
        description="Slice",
        continuous_update=False,
    )

    display(slider, widgets.interactive_output(plot, {"z": slider}))

# ============================================================
# Main visualization driver
# ============================================================

def visualize_postprocessing(
    uid,
    mode="highest_score",
    merge_by_z=True,
    interpolate=True,
):
    uid = str(uid)

    ct = load_ct(Path(img_root) / f"{uid}_0000.nii.gz")
    pred = load_label(Path(pred_root) / f"{uid}.nii.gz")
    colon = load_label(Path(colon_root) / f"{uid}.nii.gz")
    fg_probs = load_fg_probs(Path(prob_root) / f"{uid}.npz") if mode == "highest_score" else None

    # Colon debugging FIRST
    visualize_colon_debug(ct, pred, colon, "Colon mask vs prediction")

    # Step 0
    print_annotation_slices(pred, "Step 0: Original prediction")
    visualize_step(ct, pred, pred, "Step 0: Original prediction")
    # Step 1
    m1 = components_touching_colon(pred, colon)
    print_annotation_slices(m1, "Step 1: Components touching colon")
    visualize_step(ct, pred, m1, "Step 1: Components touching colon")

    # Step 2
    m2 = interpolate_missing_slices(m1) if interpolate else m1
    print_annotation_slices(m2, "Step 2: After slice interpolation")
    visualize_step(ct, m1, m2, "Step 2: After slice interpolation")

    # Step 3
    if mode == "largest":
        m3 = keep_largest_component(m2)
    else:
        m3 = keep_highest_score_component(m2, fg_probs, merge_by_z)
    
    print_annotation_slices(m3, f"Step 3: Final selection ({mode})")
    visualize_step(ct, m2, m3, f"Step 3: Final selection ({mode})")

# ============================================================
# Example
# ============================================================

if __name__ == "__main__":
    visualize_postprocessing(
        uid=4,
        mode="highest_score",
        merge_by_z=False,
        interpolate=True,
    )


run postprocessing and compute metrics:

In [ ]:
#!/usr/bin/env python3
import os
import json
import numpy as np
import nibabel as nib
import cc3d
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import seg_metrics.seg_metrics as sg

# ============================================================
# Paths
# ============================================================

pred_root    = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs_wp"
gt_root      = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/labelsTs"
colon_root   = "/data/colon_cancer/totalseg/total"
prob_root    = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs_wp"
out_mask_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs_pp"

output_json = "metrics_results_postprocessing.json"

# ============================================================
# Loading / Saving
# ============================================================

def load_nifti(path):
    img = nib.load(str(path))
    return img.get_fdata(), img.affine, img.header

def load_label(path):
    return load_nifti(path)[0].astype(np.uint8)

def save_nifti(data, affine, header, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    nib.save(
        nib.Nifti1Image(data.astype(np.uint8), affine, header),
        str(path),
    )

def load_fg_probs(npz_path):
    return np.transpose(np.load(npz_path)["probabilities"][1], (2,1,0))

# ============================================================
# Spacing helper (IMPORTANT)
# ============================================================

def get_spacing(gt_path, pred_path):
    """
    Returns spacing in (Z, Y, X) order for seg-metrics / SimpleITK.
    Prefers GT spacing, falls back to prediction spacing.
    """
    if gt_path.exists():
        hdr = nib.load(str(gt_path)).header
    else:
        hdr = nib.load(str(pred_path)).header

    spacing = hdr.get_zooms()[:3]     
    
    # DEBUG: Print spacing
    print(f"    DEBUG: Spacing from header: {spacing}")
    print(f"    DEBUG: Spacing in (Z, Y, X) order: {spacing}")
        
    return spacing

# ============================================================
# Post-processing functions (UNCHANGED)
# ============================================================

def components_touching_colon(pred_mask, colon_mask, connectivity=26):
    print(f"    DEBUG: components_touching_colon - pred_mask shape: {pred_mask.shape}, colon_mask shape: {colon_mask.shape}")
    print(f"    DEBUG: pred_mask unique values: {np.unique(pred_mask)}, colon_mask unique values: {np.unique(colon_mask)}")
    
    labeled, num = cc3d.connected_components(
        pred_mask, return_N=True, connectivity=connectivity
    )
    print(f"    DEBUG: Found {num} components in prediction mask")
    
    colon_mask_bool = colon_mask > 0
    keep_ids = [
        i for i in range(1, num + 1)
        if np.any((labeled == i) & colon_mask_bool)
    ]
    print(f"    DEBUG: Keeping {len(keep_ids)} components that touch colon")
    
    return np.isin(labeled, keep_ids).astype(np.uint8)

def interpolate_missing_slices(mask):
    """
    Interpolate missing slices along z-axis (axis=2).
    mask shape assumed to be (H, W, Z) or similar where z is axis=2.
    """
    print(f"    DEBUG: interpolate_missing_slices - input mask shape: {mask.shape}")
    
    mask = mask.astype(bool)

    # Check which z-slices contain foreground
    z_any = mask.any(axis=(0, 1))   # <-- z is axis 2
    z_idx = np.where(z_any)[0]
    
    print(f"    DEBUG: Foreground slices: {z_idx}")
    print(f"    DEBUG: Number of foreground slices: {len(z_idx)}")

    if len(z_idx) < 2:
        print("    DEBUG: Less than 2 foreground slices, skipping interpolation")
        return mask.astype(np.uint8)

    # Signed distance transforms per slice
    dt = np.zeros_like(mask, dtype=np.float32)
    for z in range(mask.shape[2]):
        fg = mask[:, :, z]
        dt[:, :, z] = (
            distance_transform_edt(~fg) -
            distance_transform_edt(fg)
        )

    repaired = mask.copy()
    for z in range(z_idx[0] + 1, z_idx[-1]):
        if not z_any[z]:
            zp = z_idx[z_idx < z][-1]
            zn = z_idx[z_idx > z][0]
            w = (z - zp) / float(zn - zp)

            repaired[:, :, z] = (
                (1 - w) * dt[:, :, zp] + w * dt[:, :, zn]
            ) < 0

    print(f"    DEBUG: After interpolation - repaired shape: {repaired.shape}")
    print(f"    DEBUG: Repaired foreground slices: {np.where(repaired.any(axis=(0,1)))[0]}")
    
    return repaired.astype(np.uint8)


def connected_components(mask):
    print(f"    DEBUG: connected_components - input mask shape: {mask.shape}")
    labeled = cc3d.connected_components(mask)
    num_components = labeled.max()
    print(f"    DEBUG: Found {num_components} connected components")
    return [(labeled == cid) for cid in range(1, labeled.max() + 1)]

def merge_components_by_z_overlap(components):
    print(f"    DEBUG: merge_components_by_z_overlap - input {len(components)} components")
    merged, used = [], [False] * len(components)

    # z is axis=2
    z_ranges = [
        (np.where(c)[2].min(), np.where(c)[2].max())
        for c in components
        if c.any()
    ]
    
    print(f"    DEBUG: Component z-ranges: {z_ranges}")

    for i, (c, (z0, z1)) in enumerate(zip(components, z_ranges)):
        if used[i]:
            continue

        m = c.copy()
        used[i] = True

        for j, (c2, (zz0, zz1)) in enumerate(zip(components, z_ranges)):
            if not used[j] and not (zz1 < z0 or zz0 > z1):
                print(f"    DEBUG: Merging components {i} and {j} (z-overlap)")
                m |= c2
                used[j] = True

        merged.append(m)
    
    print(f"    DEBUG: After merging - {len(merged)} components remain")

    return merged


def keep_largest_component(mask):
    print(f"    DEBUG: keep_largest_component - input mask shape: {mask.shape}")
    labeled = cc3d.connected_components(mask)
    if labeled.max() == 0:
        print("    DEBUG: No components found")
        return mask
    sizes = cc3d.statistics(labeled)["voxel_counts"]
    largest_idx = np.argmax(sizes[1:]) + 1
    print(f"    DEBUG: Largest component is {largest_idx} with {sizes[largest_idx]} voxels")
    return (labeled == largest_idx).astype(np.uint8)

def keep_highest_score_component(mask, fg_probs, merge_by_z):
    if not mask.any():
        print( "    DEBUG: empty mask → skipping highest-score selection")
        return mask.astype(np.uint8)
    print(f"    DEBUG: keep_highest_score_component - mask shape: {mask.shape}, fg_probs shape: {fg_probs.shape}")
    comps = connected_components(mask)
    if merge_by_z and len(comps) > 1:
        comps = merge_components_by_z_overlap(comps)
    scores = [fg_probs[c].mean() if c.any() else -np.inf for c in comps]
    best_idx = np.argmax(scores)
    print(f"    DEBUG: Component scores: {scores}")
    print(f"    DEBUG: Selected component {best_idx} with score {scores[best_idx]:.4f}")
    return comps[best_idx].astype(np.uint8)

# ============================================================
# seg-metrics wrapper
# ============================================================

METRICS = [
    "dice",
    "precision",
    "recall",
    "fpr",
    "fnr",
    "msd",
    "hd95",
]

def compute_metrics(pred, gt, spacing):
    print(f"    DEBUG: compute_metrics - pred shape: {pred.shape}, gt shape: {gt.shape}, spacing: {spacing}")
    print(f"    DEBUG: pred unique values: {np.unique(pred)}, gt unique values: {np.unique(gt)}")
    print(f"    DEBUG: pred foreground voxels: {np.sum(pred > 0)}, gt foreground voxels: {np.sum(gt > 0)}")
    
    metrics = sg.write_metrics(
        labels=[1],
        pred_img=pred.astype(np.uint8),
        gdth_img=gt.astype(np.uint8),
        metrics=METRICS,
        spacing=spacing,
    )
    return {m: float(metrics[0][m][0]) for m in METRICS}

# ============================================================
# Main evaluation
# ============================================================

def evaluate(
    use_colon_filter=True,
    use_interpolation=True,
    final_selection="highest_score",
    merge_by_z=True,
    save_masks=False,
    debug_single_case=False,
    debug_case_id=None,
):
    results = {}
    before_all, after_all = [], []

    # Get list of prediction files
    pred_files = sorted([f for f in os.listdir(pred_root) if f.endswith((".nii", ".nii.gz"))])
    
    if debug_single_case and debug_case_id:
        # Filter to only the debug case
        pred_files = [f for f in pred_files if debug_case_id in f]
        if not pred_files:
            print(f"ERROR: Case {debug_case_id} not found in {pred_root}")
            return
        print(f"DEBUG MODE: Processing only case {debug_case_id}")
    
    for pred_file in pred_files:
        uid = pred_file.replace(".nii.gz", "").replace(".nii", "")
        print("\n" + "="*60)
        print(f"▶ Processing {uid}")
        print("="*60)

        pred_path = Path(pred_root) / pred_file
        gt_path   = Path(gt_root) / pred_file

        # Load data
        print(f"  Loading prediction from: {pred_path}")
        pred, aff, hdr = load_nifti(pred_path)
        print(f"    DEBUG: Loaded pred shape: {pred.shape}, dtype: {pred.dtype}")
        
        print(f"  Loading ground truth from: {gt_path}")
        gt = load_label(gt_path)
        print(f"    DEBUG: Loaded gt shape: {gt.shape}, dtype: {gt.dtype}")

        spacing = get_spacing(gt_path, pred_path)
        print(f"    DEBUG: Using spacing: {spacing}")

        pred = pred.astype(np.uint8)
        print(f"    DEBUG: After conversion - pred shape: {pred.shape}, unique: {np.unique(pred)}")

        # Compute metrics before processing
        print("\n  Computing metrics BEFORE post-processing:")
        metrics_before = compute_metrics(pred, gt, spacing)

        # ---------- Postprocessing ----------
        print("\n  Starting post-processing pipeline:")
        m = pred.copy()

        if use_colon_filter:
            colon_path = Path(colon_root) / f"{uid}.nii.gz"
            print(f"  Loading colon mask from: {colon_path}")
            colon = load_label(colon_path)
            print(f"    DEBUG: Colon mask shape: {colon.shape}, unique: {np.unique(colon)}")
            m = components_touching_colon(m, colon)
            print(f"    DEBUG: After colon filter - mask shape: {m.shape}, foreground voxels: {np.sum(m > 0)}")

        if use_interpolation:
            m = interpolate_missing_slices(m)
            print(f"    DEBUG: After interpolation - mask shape: {m.shape}, foreground voxels: {np.sum(m > 0)}")

        if final_selection == "largest":
            print("  Selecting largest component")
            m = keep_largest_component(m)
            print(f"    DEBUG: After largest component - mask shape: {m.shape}, foreground voxels: {np.sum(m > 0)}")
        elif final_selection == "highest_score":
            prob_path = Path(prob_root) / f"{uid}.npz"
            print(f"  Loading probabilities from: {prob_path}")
            fg_probs = load_fg_probs(prob_path)
            print(f"    DEBUG: Loaded fg_probs shape: {fg_probs.shape}")
            m = keep_highest_score_component(m, fg_probs, merge_by_z)
            print(f"    DEBUG: After highest score - mask shape: {m.shape}, foreground voxels: {np.sum(m > 0)}")

        # Compute metrics after processing
        print("\n  Computing metrics AFTER post-processing:")
        metrics_after = compute_metrics(m, gt, spacing)

        if save_masks:
            out_path = Path(out_mask_dir) / pred_file
            print(f"  Saving processed mask to: {out_path}")
            save_nifti(m, aff, hdr, out_path)

        results[uid] = {
            "before": metrics_before,
            "after": metrics_after,
            "delta": {k: metrics_after[k] - metrics_before[k] for k in METRICS},
        }

        before_all.append(metrics_before)
        after_all.append(metrics_after)

        print(f"\n  Results for {uid}:")
        print(f"    Dice: {metrics_before['dice']:.4f} → {metrics_after['dice']:.4f}")
        print(f"    Precision: {metrics_before['precision']:.4f} → {metrics_after['precision']:.4f}")
        print(f"    Recall: {metrics_before['recall']:.4f} → {metrics_after['recall']:.4f}")
        
        # If debugging single case, break after processing
        if debug_single_case:
            print("\n" + "="*60)
            print("DEBUG MODE: Stopping after single case")
            print("="*60)
            break

    if not debug_single_case or len(before_all) > 0:
        summary = {
            "mean_before": {m: float(np.mean([x[m] for x in before_all])) for m in METRICS},
            "mean_after":  {m: float(np.mean([x[m] for x in after_all])) for m in METRICS},
            "mean_delta": {
                m: float(np.mean([a[m] - b[m] for a, b in zip(after_all, before_all)]))
                for m in METRICS
            },
        }

        with open(output_json, "w") as f:
            json.dump({"per_case": results, "summary": summary}, f, indent=4)

        print("\n✅ Done")
        print(json.dumps(summary, indent=2))

# ============================================================
# Run
# ============================================================

if __name__ == "__main__":
    # For debugging a single case, set debug_single_case=True and provide case ID
    evaluate(
        use_colon_filter=True,
        use_interpolation=True,
        final_selection="highest_score",
        merge_by_z=True,
        save_masks=True,
        debug_single_case=False,  # Set to True to debug only one case
        debug_case_id="18.nii.gz",  # Replace with actual case ID to debug
    )